In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install sentence-transformers --quiet

import pandas as pd
import numpy as np
import torch
import ast
import os
import json
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

print(f"GPU disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB


In [2]:
INPUT_PATH = '/kaggle/input/datasets/catalinalupu/movies-ready-for-summarization/movies-ready-for-summarization.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f"Filme incarcate: {len(df):,}")
print(f"Coloane: {df.columns.tolist()}")

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review']


In [3]:
def build_doc_text(row):
    parts = []
    if pd.notna(row.get('title')) and str(row['title']).strip():
        parts.append(str(row['title']).strip() + '.')
    if pd.notna(row.get('overview')) and str(row['overview']).strip():
        parts.append(str(row['overview']).strip())
    if pd.notna(row.get('genres')) and str(row['genres']).strip():
        parts.append('Genres: ' + str(row['genres']).strip() + '.')
    if pd.notna(row.get('keywords')) and str(row['keywords']).strip():
        kw = str(row['keywords']).strip()
        parts.append('Keywords: ' + kw[:250] + '.')
    if pd.notna(row.get('cast')) and str(row['cast']).strip():
        cast = str(row['cast']).strip()
        parts.append('Cast: ' + cast[:200] + '.')
    return ' '.join(parts)

df['doc_text'] = df.apply(build_doc_text, axis=1)
df['tagline']  = df['tagline'].fillna('').astype(str).str.strip()

mask = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f"Filme cu tagline si doc_text: {len(df_valid):,}")
print(f"Filme eliminate (tagline sau doc lipsa): {len(df) - len(df_valid):,}")

print(f"\nExemplu doc_text pentru '{df_valid.iloc[0]['title']}':")
print(df_valid.iloc[0]['doc_text'][:400])
print(f"\nTagline: {df_valid.iloc[0]['tagline']}")

Filme cu tagline si doc_text: 40,197
Filme eliminate (tagline sau doc lipsa): 0

Exemplu doc_text pentru 'Toy Story':
Toy Story. Led by Woody, Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Genres: ['Family', 'Comedy', 'Animation', 'Adventure']. Keywords: ['rescue', 'friendship

Tagline: The adventure takes off when toys come to life!


In [4]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')

sample = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample]

print(f"Tokeni doc_text — Medie: {np.mean(lengths):.1f}")
print(f"Tokeni doc_text — Max:   {max(lengths)}")
print(f"Tokeni doc_text — 99%:   {np.percentile(lengths, 99):.1f}")
print(f"Trunchiate la 512:       {sum(l > 512 for l in lengths)}/1000 ({sum(l > 512 for l in lengths)/10:.1f}%)")

Tokeni doc_text — Medie: 220.1
Tokeni doc_text — Max:   374
Tokeni doc_text — 99%:   336.0
Trunchiate la 512:       0/1000 (0.0%)


In [5]:
from sklearn.model_selection import train_test_split
from sentence_transformers import InputExample

train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
print(f"Training: {len(train_df):,} filme")
print(f"Eval:     {len(eval_df):,} filme")

train_examples = [
    InputExample(texts=[row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(row['tagline'].split()) >= 3  
]

print(f"\nExemple de antrenare: {len(train_examples):,}")
print(f"\nExemplu pereche:")
print(f"  Query (tagline): {train_examples[0].texts[0]}")
print(f"  Doc:             {train_examples[0].texts[1][:200]}...")

Training: 36,177 filme
Eval:     4,020 filme

Exemple de antrenare: 35,788

Exemplu pereche:
  Query (tagline): She loved and trusted him until he cut off her head.
  Doc:             Savage Intruder. An enigmatic young man manipulates his way into working at the decaying mansion of a once prolific, but now reclusive and alcoholic, movie star named Katharine Packard. While the rest...


In [6]:
MODEL_NAME   = 'all-mpnet-base-v2'
OUTPUT_MODEL = OUTPUT_PATH + 'sbert_finetuned'
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

sbert = SentenceTransformer(MODEL_NAME, device=device)
print(f"Model {MODEL_NAME} incarcat pe {device}")

dim = sbert.encode(['test']).shape[1]
print(f"Dimensiune embedding: {dim}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model all-mpnet-base-v2 incarcat pe cuda
Dimensiune embedding: 768


In [7]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

if isinstance(sbert, torch.nn.DataParallel):
    sbert = sbert.module

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sbert  = sbert.to(device)

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f"Device: {device} | Total: {total_mem:.1f} GB | Libera: {free_mem:.2f} GB")

_tok = sbert.tokenizer

def tokenize(texts):
    return _tok(texts, padding=True, truncation=True, max_length=256, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    return out['sentence_embedding']

def mnr_loss(emb_a, emb_p, scale=20.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE  = 16
ACCUM_STEPS = 4
EPOCHS      = 3

train_dl     = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=simple_collate)
total_steps  = (len(train_dl) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f"Antrenare: {len(train_examples):,} exemple | batch={BATCH_SIZE} x accum={ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS} efectiv | epoci={EPOCHS}")

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl), desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, (anchors, positives) in pbar:
        enc_a = tokenize(anchors)
        enc_p = tokenize(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss(emb_a, emb_p) / ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_loss += loss.item()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += accum_loss
            pbar.set_postfix({'loss': f'{accum_loss:.4f}'})
            accum_loss = 0.0

    steps_done = len(train_dl) // ACCUM_STEPS
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss mediu: {total_loss / steps_done:.4f}")

sbert.save(OUTPUT_MODEL)
print(f"\nFine-tuning complet! Model salvat in: {OUTPUT_MODEL}")

Device: cuda:0 | Total: 14.6 GB | Libera: 14.15 GB
Antrenare: 35,788 exemple | batch=16 x accum=4 = 64 efectiv | epoci=3


Epoch 1/3:   0%|          | 0/2237 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 1.0342


Epoch 2/3:   0%|          | 0/2237 [00:00<?, ?it/s]

Epoch 2/3 — Loss mediu: 0.7351


Epoch 3/3:   0%|          | 0/2237 [00:00<?, ?it/s]

Epoch 3/3 — Loss mediu: 0.5945


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning complet! Model salvat in: /kaggle/working/sbert_finetuned


In [8]:
sbert_base = SentenceTransformer(MODEL_NAME,   device=device)  
sbert_ft   = SentenceTransformer(OUTPUT_MODEL, device=device)  

queries = df_valid['tagline'].tolist()
docs    = df_valid['doc_text'].tolist()
BATCH_SIZE_EMB = 256

print("Generare embeddings baseline (all-mpnet-base-v2 pretrained)...")
q_emb_base = sbert_base.encode(queries, batch_size=BATCH_SIZE_EMB,
                                show_progress_bar=True, convert_to_numpy=True)
d_emb_base = sbert_base.encode(docs,    batch_size=BATCH_SIZE_EMB,
                                show_progress_bar=True, convert_to_numpy=True)
print(f"Baseline: query={q_emb_base.shape}, doc={d_emb_base.shape}")

print("\nGenerare embeddings fine-tuned...")
q_emb_ft = sbert_ft.encode(queries, batch_size=BATCH_SIZE_EMB,
                             show_progress_bar=True, convert_to_numpy=True)
d_emb_ft = sbert_ft.encode(docs,    batch_size=BATCH_SIZE_EMB,
                             show_progress_bar=True, convert_to_numpy=True)
print(f"Fine-tuned: query={q_emb_ft.shape}, doc={d_emb_ft.shape}")

np.save(OUTPUT_PATH + 'q_emb_base.npy', q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy', d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_ft.npy',  q_emb_ft)
np.save(OUTPUT_PATH + 'd_emb_ft.npy',  d_emb_ft)
print("\nEmbeddings salvate!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings baseline (all-mpnet-base-v2 pretrained)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Baseline: query=(40197, 768), doc=(40197, 768)

Generare embeddings fine-tuned...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Fine-tuned: query=(40197, 768), doc=(40197, 768)

Embeddings salvate!


In [9]:
def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]

df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f"Genuri unice: {len(genre_to_indices)}")
print(f"Exemple genuri: {list(genre_to_indices.keys())[:10]}")

sizes = [len(v) for v in genre_to_indices.values()]
print(f"\nFilme per gen — Medie: {np.mean(sizes):.0f}, Min: {min(sizes)}, Max: {max(sizes)}")

pool_sizes = []
for i in range(min(1000, len(df_valid))):
    pool = set()
    for g in df_valid['genres_list'].iloc[i]:
        pool.update(genre_to_indices.get(g, []))
    pool_sizes.append(len(pool))
print(f"Pool genre-filtered — Medie: {np.mean(pool_sizes):.0f} filme per query")

Genuri unice: 19
Exemple genuri: ['family', 'comedy', 'animation', 'adventure', 'fantasy', 'romance', 'drama', 'action', 'thriller', 'crime']

Filme per gen — Medie: 4774, Min: 1297, Max: 17268
Pool genre-filtered — Medie: 18487 filme per query


In [11]:
def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20], batch_size=512):
    N = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)  

        top_k_idx = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k_idx[i][np.argsort(scores[i][top_k_idx[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 5000 == 0:
            print(f"  {start}/{N}...")

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30):
    np.random.seed(42)
    indices = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits = {k: 0 for k in k_values}
    pool_size_list = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx) 

        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr = np.array(sorted(pool))
        pool_size_list.append(len(pool_arr))

        q      = query_emb[idx:idx+1]
        d_pool = doc_emb[pool_arr]
        scores = cosine_similarity(q, d_pool)[0]

        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1  # 1-indexed

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f"  Pool mediu per query: {np.mean(pool_size_list):.0f} filme")
    return {k: hits[k] / n for k in k_values}

In [12]:
results = {}

print("FULL POOL (toti filmele)")

print("\nBaseline (all-mpnet-base-v2, fara fine-tuning):")
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base)
print(f"  {results['base_full']}")

print("\nFine-tuned (all-mpnet-base-v2, cu fine-tuning):")
results['ft_full'] = evaluate_full_pool(q_emb_ft, d_emb_ft)
print(f"  {results['ft_full']}")

print("\nGENRE-FILTERED POOL (filme cu acelasi gen)")

print("\nBaseline (all-mpnet-base-v2, fara fine-tuning):")
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f"  {results['base_genre']}")

print("\nFine-tuned (all-mpnet-base-v2, cu fine-tuning):")
results['ft_genre'] = evaluate_genre_filtered(q_emb_ft, d_emb_ft)
print(f"  {results['ft_genre']}")

with open(OUTPUT_PATH + 'results_fixed.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nRezultate salvate!")

FULL POOL (toti filmele)

Baseline (all-mpnet-base-v2, fara fine-tuning):
  0/40197...
  {1: 0.06445754658307834, 3: 0.09819140731895415, 5: 0.11533198994949873, 10: 0.14244844142597707, 20: 0.17414234893151231}

Fine-tuned (all-mpnet-base-v2, cu fine-tuning):
  0/40197...
  {1: 0.10023136054929473, 3: 0.15521058785481504, 5: 0.18702888275244423, 10: 0.2362614125432246, 20: 0.2934796129064358}

GENRE-FILTERED POOL (filme cu acelasi gen)

Baseline (all-mpnet-base-v2, fara fine-tuning):
  Pool mediu per query: 15835 filme
  {1: 0.08466666666666667, 3: 0.121, 5: 0.14, 10: 0.17433333333333334, 20: 0.21133333333333335}

Fine-tuned (all-mpnet-base-v2, cu fine-tuning):
  Pool mediu per query: 15835 filme
  {1: 0.11366666666666667, 3: 0.17066666666666666, 5: 0.20066666666666666, 10: 0.24733333333333332, 20: 0.30566666666666664}

Rezultate salvate!


In [13]:
print(f"{'Model':<45} {'Hit@1':>7} {'Hit@3':>7} {'Hit@5':>7} {'Hit@10':>7} {'Hit@20':>7}")

labels = [
    ('base_full',  'Baseline   | full pool   (~37k filme)'),
    ('ft_full',    'Fine-tuned | full pool   (~37k filme)'),
    ('base_genre', 'Baseline   | genre pool  (~avg filme)'),
    ('ft_genre',   'Fine-tuned | genre pool  (~avg filme)'),
]

for key, label in labels:
    r = results[key]
    print(f"{label:<45} {r[1]:>7.3f} {r[3]:>7.3f} {r[5]:>7.3f} {r[10]:>7.3f} {r[20]:>7.3f}")

print()
gain_1  = (results['ft_genre'][1]  - results['base_full'][1])  / results['base_full'][1]  * 100
gain_10 = (results['ft_genre'][10] - results['base_full'][10]) / results['base_full'][10] * 100
print(f"Castig fine-tuned genre vs baseline full — Hit@1: +{gain_1:.1f}%, Hit@10: +{gain_10:.1f}%")

Model                                           Hit@1   Hit@3   Hit@5  Hit@10  Hit@20
Baseline   | full pool   (~37k filme)           0.064   0.098   0.115   0.142   0.174
Fine-tuned | full pool   (~37k filme)           0.100   0.155   0.187   0.236   0.293
Baseline   | genre pool  (~avg filme)           0.085   0.121   0.140   0.174   0.211
Fine-tuned | genre pool  (~avg filme)           0.114   0.171   0.201   0.247   0.306

Castig fine-tuned genre vs baseline full — Hit@1: +76.3%, Hit@10: +73.6%


In [14]:
def recommend(query: str, top_k: int = 5, model='finetuned'):
    m = sbert_ft if model == 'finetuned' else sbert_base
    d = d_emb_ft  if model == 'finetuned' else d_emb_base

    q_emb  = m.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, d)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"\nQuery: \"{query}\"")
    print(f"{'Rang':<5} {'Titlu':<40} {'Gen':<30} {'Scor':>6}")
    print('-' * 85)
    for rank, idx in enumerate(top_idx, 1):
        row    = df_valid.iloc[idx]
        genres = str(row['genres'])[:28]
        print(f"{rank:<5} {row['title']:<40} {genres:<30} {scores[idx]:>6.3f}")


recommend("animated toys that come to life and go on adventures", top_k=5)
recommend("a superhero saves the world from an alien invasion", top_k=5)
recommend("romantic comedy set in New York with funny misunderstandings", top_k=5)
recommend("psychological thriller about dreams and reality", top_k=5)


Query: "animated toys that come to life and go on adventures"
Rang  Titlu                                    Gen                              Scor
-------------------------------------------------------------------------------------
1     Raggedy Ann & Andy: A Musical Adventure! ['Animation', 'Adventure', '    0.545
2     Toy Story                                ['Family', 'Comedy', 'Animat    0.543
3     The Christmas Toy                        ['Family', 'Comedy', 'Fantas    0.540
4     Toopy and Binoo The Movie                ['Animation', 'Adventure', '    0.540
5     Patch Town                               ['Music', 'Adventure', 'Come    0.516

Query: "a superhero saves the world from an alien invasion"
Rang  Titlu                                    Gen                              Scor
-------------------------------------------------------------------------------------
1     Alien Warrior                            ['Crime', 'Science Fiction',    0.672
2     Shin Ultraman     

In [15]:
query = "animated toys that come to life and go on adventures"

print("BASELINE")
recommend(query, top_k=5, model='baseline')

print("\nFINE-TUNED")
recommend(query, top_k=5, model='finetuned')

BASELINE

Query: "animated toys that come to life and go on adventures"
Rang  Titlu                                    Gen                              Scor
-------------------------------------------------------------------------------------
1     Plastic Galaxy: The Story of Star Wars Toys ['Documentary']                 0.512
2     Fantastic Animation Festival             ['Animation']                   0.510
3     Toy Story                                ['Family', 'Comedy', 'Animat    0.509
4     The Christmas Toy                        ['Family', 'Comedy', 'Fantas    0.503
5     Eclectic Shorts by Eric Leiser           ['Animation', 'Fantasy', 'Ad    0.503

FINE-TUNED

Query: "animated toys that come to life and go on adventures"
Rang  Titlu                                    Gen                              Scor
-------------------------------------------------------------------------------------
1     Raggedy Ann & Andy: A Musical Adventure! ['Animation', 'Adventure', '    0.54